# Example notebook to run assesment of NZ challenge submission

This will run the assesement algorithms on an NZ challenge submission

#### Standard imports

In [ ]:
import tables_io, qp
import numpy as np
import os
import matplotlib.pyplot as plt
from nz_data_challenge import submit_utils, metrics, utils, evaluation
from pathlib import Path

### Paths

In [ ]:
submission_name = 'rail_knn_4tasks'
test_only = True
if test_only:
    submit_dir = f'submission_test/{submission_name}'
    results_dir = f'results_test/{submission_name}'
    test_suffix = 'ddf_01'
    truth_dir = '../public'
else:
    submit_dir = f'submission/{submission_name}'
    results_dir = f'results/{submission_name}'
    test_suffix = 'wfd'    
    truth_dir = '../reserved'

model_dir = f'../models/{submission_name}'
public_dir = '../public'

try:
    os.makedirs(submit_dir)
except:
    pass
try:
    os.makedirs(results_dir)
except:
    pass

### Filepaths, getting data

In [ ]:
taskset = 'taskset_1'
sim = 'cardinal'
scenario = '1yr'
wfd_file = f"{public_dir}/nz_challenge_{taskset}_{sim}_{scenario}_{test_suffix}.hdf5"
truth_file = f'{truth_dir}/nz_challenge_{taskset}_{sim}_{scenario}_{test_suffix}.hdf5'
nz_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_estimate_{test_suffix}.hdf5"
bhat_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_bhat_{test_suffix}.hdf5"

grid_edges = utils.Z_BIN_EDGES[taskset]
grid_centers = 0.5*(grid_edges[0:-1]+grid_edges[1:])
n_grid_points = len(grid_centers)

tomo_bin_edges = utils.TOMO_BIN_EDGES[taskset]
tomo_bin_centers = 0.5*(tomo_bin_edges[0:-1]+tomo_bin_edges[1:])
n_tomo_bins = len(tomo_bin_edges) - 1
plot_bin_edges = np.linspace(-0.5,n_tomo_bins-0.5,n_tomo_bins+1) 

nz_estimates = qp.read(nz_file)
test_data = tables_io.read(wfd_file)
truth = tables_io.read(truth_file)
true_redshifts = truth['redshift']
bhat_data = tables_io.read(bhat_file)
bin_assignments = np.squeeze(bhat_data['tomo_bin_index'])
hist_list = []
true_assignments = utils.get_true_bin_assignments(true_redshifts, tomo_bin_edges)

### Getting distibutions

In [ ]:
nz_distributions = utils.get_nz_distributions(nz_estimates, grid_centers, n_tomo_bins)
true_distributions = utils.get_true_nz_distributions(true_redshifts, bin_assignments, grid_edges, n_tomo_bins)

### Check that files contain what they should

In [ ]:
if test_only:
    test_ids = set(bhat_data['object_id'].astype(int))
else:
    test_ids = set(np.arange(0, 1_000_000).astype(int))
submit_utils.check_files(nz_file, bhat_file, test_ids, n_tomo_bins)

### Compute bin assignment metrics

In [ ]:
assignment_metrics = evaluation.evaluate_bin_assignments(true_assignments, bin_assignments)
assignment_metrics

### Compute distributions metrics

In [ ]:
nz_metrics = evaluation.evaluate_distributions(true_distributions, nz_distributions, grid_edges, nz_estimates.ancil['n_objects'])
nz_metrics

### Draw the confusion matrix

In [ ]:
fig_confusion = evaluation.plot_confusion_matrix(true_assignments, bin_assignments, n_tomo_bins)

### Draw the n(z) distritubions

In [ ]:
fig_nz = evaluation.plot_nz_data(true_distributions, nz_distributions, grid_edges)

In [ ]:
fig_mean_and_rms = evaluation.plot_nz_mean_and_rms(true_distributions, nz_distributions, grid_edges)